In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.special import erfcinv
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import LogLocator, MaxNLocator

# ----------------------------
# Matplotlib style (match "left top" panel vibe)
# ----------------------------
plt.style.use("default")

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    # "savefig.facecolor": "white",
    # "savefig.transparent": False,

    # Font / text
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    # Axes / ticks
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.minor.width": 0.8,
    "ytick.minor.width": 0.8,

    # Grid (subtle)
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.8,

    # Lines / markers
    "lines.linewidth": 1.6,
    "lines.markersize": 5,

    # Figure
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})

In [3]:
# ----------------------------
# Helpers
# ----------------------------
def sci_times_formatter(skip_exp=0):
    def _fmt(y, _):
        if y == 0:
            return "0"
        exp = int(np.floor(np.log10(abs(y))))
        if exp == skip_exp:
            return f"{y:.2f}"
        mant = y / (10 ** exp)
        return rf"${mant:.1f}\times10^{{{exp}}}$"
    return FuncFormatter(_fmt)


def q_from_ber(ber: np.ndarray) -> np.ndarray:
    """
    Q = 20 log10( sqrt(2) * erfc^{-1}(2*BER) )
    """
    ber = np.asarray(ber, dtype=float)
    ber = np.clip(ber, 1e-300, 0.499999999)  # keep erfcinv stable
    return 20.0 * np.log10(np.sqrt(2.0) * erfcinv(2.0 * ber))


def infer_x_column(df: pd.DataFrame) -> str:
    # Prefer common names; otherwise take the first numeric column
    preferred = [
        "Average Power [dBm]", "Average Power (dBm)", "Average Power", "P_avg (dBm)",
        "Power [dBm]", "Power (dBm)", "Power", "P (dBm)", "P"
    ]
    cols = list(df.columns)
    for c in preferred:
        if c in cols:
            return c
    # Fallback: first numeric column
    for c in cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            return c
    return cols[0]


def extract_series_with_errors(df: pd.DataFrame, xcol: str):
    """
    Returns list of (label, y, yerr_or_None).
    Error columns are detected by suffix patterns.
    """
    error_suffixes = ("_err", "_stderr", "_se", "_std", "_sigma", " err", " stderr", " se", " std")
    cols = [c for c in df.columns if c != xcol]

    # Identify error columns
    err_cols = set()
    for c in cols:
        lc = c.lower()
        if any(lc.endswith(sfx) for sfx in error_suffixes):
            err_cols.add(c)

    # Map each "value" col to its error col (if present)
    series = []
    for c in cols:
        if c in err_cols:
            continue

        # only take numeric BER columns
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue

        # try match error column
        c_l = c.lower()
        candidates = []
        for e in err_cols:
            e_l = e.lower()
            # common patterns: "{c}_err", "{c} err"
            if e_l.startswith(c_l) or e_l.replace("_", " ").startswith(c_l):
                candidates.append(e)
        yerr = df[candidates[0]].to_numpy(dtype=float) if candidates else None

        series.append((c, df[c].to_numpy(dtype=float), yerr))

    return series


def should_use_logy(y_arrays):
    y = np.concatenate([np.asarray(a, float).ravel() for a in y_arrays if a is not None])
    y = y[np.isfinite(y)]
    y = y[y > 0]
    if y.size < 2:
        return False
    return (y.max() / y.min()) > 20.0


# Map filename (lowercase, without path) → figure title
TITLE_MAP = {
    "64qam rings.csv": "64-QAM 5 WDM channels 10×100 km propagation",
    "ber_power(figure 4).csv": "16-QAM single channel 15×80 km",
    "ber_vs_power_single_channel_10x100km_final.csv": "16-QAM single channel 10×100 km",
    "ber_vs_average_power_11_channels_10x100km_final.csv": "16-QAM 11 WDM channels 10×100 km",
}

def title_from_filename(csv_path):
    name = os.path.basename(csv_path).lower()
    return TITLE_MAP.get(name, "16-QAM 10×100 km propagation")



def plot_metric(
    x, series, ylabel, title, out_pdf,
    y_transform=None, yerr_transform=None,
    logy="auto", q_factor=False
):
    fig, ax = plt.subplots(figsize=(6, 4))
    if not q_factor:
        ax.yaxis.set_major_formatter(sci_times_formatter(skip_exp=0))



    markers = ["o", "s", "^", "D", "v", "P", "X", "<", ">", "h", "*"]
    for i, (label, y, yerr) in enumerate(series):
        yy = y_transform(y) if y_transform else y
        ee = None
        if yerr is not None:
            ee = yerr_transform(y, yerr) if yerr_transform else yerr

        ax.errorbar(
            x, yy, yerr=ee,
            marker=markers[i % len(markers)],
            capsize=2.5, elinewidth=1.0, label=label
        )

    ax.set_title(title)
    ax.set_xlabel("Average Power [dBm]")
    ax.set_ylabel(ylabel)

    if logy == "auto":
        logy = should_use_logy([y_transform(y) if y_transform else y for _, y, _ in series])
    if logy:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(numticks=7))
        ax.yaxis.set_minor_locator(LogLocator(subs="auto", numticks=7))
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=7))


    ax.legend(frameon=True, fancybox=False, framealpha=0.9)
    ax.grid(True, which="both")
    fig.tight_layout()
    fig.savefig(out_pdf, format="pdf")
    plt.close(fig)


# ----------------------------
# Main: generate BER + Q PDFs for each CSV
# ----------------------------
# Point this to your folder, or keep current directory:
CSV_DIR = "."  # e.g. "/mnt/data" or "." if running locally
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {os.path.abspath(CSV_DIR)}")

for csv_path in csv_files:
    df = pd.read_csv(csv_path)

    # Drop completely empty columns, strip column names
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(axis=1, how="all")

    xcol = infer_x_column(df)
    x = df[xcol].to_numpy(dtype=float)

    # Sort by x (nice curves)
    order = np.argsort(x)
    x = x[order]
    df = df.iloc[order].reset_index(drop=True)

    series = extract_series_with_errors(df, xcol)
    if not series:
        print(f"[SKIP] No numeric series found in {os.path.basename(csv_path)}")
        continue

    base = os.path.splitext(os.path.basename(csv_path))[0]
    out_ber = os.path.join(os.path.dirname(csv_path), f"{base}_BER.pdf")
    out_q = os.path.join(os.path.dirname(csv_path), f"{base}_Q.pdf")

    # --- BER plot (auto log scale if wide dynamic range) ---
    plot_metric(
        x=x,
        series=series,
        ylabel="Bit Error Rate (BER)",
        title=title_from_filename(csv_path),
        out_pdf=out_ber,
        y_transform=None,
        yerr_transform=None,
        logy="auto",
    )

    # --- Q plot (linear) ---
    def q_err_propagation(ber, ber_err):
        # dQ/dBER = 20/ln(10) * (1 / (sqrt(2)*erfcinv(2BER))) * d/dBER[ sqrt(2)*erfcinv(2BER) ]
        # derivative of erfcinv(u): d/du erfcinv(u) = -sqrt(pi)/2 * exp((erfcinv(u))^2)
        # u = 2BER, so du/dBER = 2
        ber = np.asarray(ber, float)
        ber_err = np.asarray(ber_err, float)
        ber_c = np.clip(ber, 1e-300, 0.499999999)

        z = erfcinv(2.0 * ber_c)  # z = erfcinv(u)
        # dy/dBER where y = sqrt(2)*z
        # dz/dBER = dz/du * du/dBER = ( -sqrt(pi)/2 * exp(z^2) ) * 2 = -sqrt(pi) * exp(z^2)
        dz_dber = -np.sqrt(np.pi) * np.exp(z * z)
        dy_dber = np.sqrt(2.0) * dz_dber

        y = np.sqrt(2.0) * z
        dQ_dber = (20.0 / np.log(10.0)) * (1.0 / np.clip(y, 1e-300, np.inf)) * dy_dber
        return np.abs(dQ_dber) * ber_err

    plot_metric(
        x=x,
        series=series,
        ylabel="Q-factor [dB]",
        title="Q-factor vs Power",
        out_pdf=out_q,
        y_transform=q_from_ber,
        yerr_transform=q_err_propagation,
        logy=False,
        q_factor=True
    )

    print(f"[OK] {os.path.basename(csv_path)} ->\n  {out_ber}\n  {out_q}")


[OK] 64qam rings.csv ->
  ./64qam rings_BER.pdf
  ./64qam rings_Q.pdf
[OK] BER_Power(Figure 4).csv ->
  ./BER_Power(Figure 4)_BER.pdf
  ./BER_Power(Figure 4)_Q.pdf
[OK] BER_vs_Average_Power_11_Channels_10x100km_final.csv ->
  ./BER_vs_Average_Power_11_Channels_10x100km_final_BER.pdf
  ./BER_vs_Average_Power_11_Channels_10x100km_final_Q.pdf
[OK] BER_vs_Power_Single_Channel_10x100km_final.csv ->
  ./BER_vs_Power_Single_Channel_10x100km_final_BER.pdf
  ./BER_vs_Power_Single_Channel_10x100km_final_Q.pdf
